# Realtime inference

Define a Kafka consumer to subscribe to messages in the feature store

In [5]:
from confluent_kafka import Consumer
import json

def fetch_all_feature_records():
    # Kafka Consumer configuration for reading from the beginning of the topic
    conf = {
        'bootstrap.servers': "localhost:9092",
        'group.id': "feature_store_reader",
        'auto.offset.reset': 'latest'
        }

    # Initialize Kafka Consumer and subscribe to the topic
    consumer = Consumer(conf)
    consumer.subscribe(['feature_store'])

    feature_records = []  # List to store feature data

    try:
        while True:
            msg = consumer.poll(1.0)  # Poll for messages with a 1-second timeout
            if msg is None:
                break  # Exit loop if no more messages
            if not msg.error():
                # Convert message from JSON and add to list
                feature_records.append(json.loads(msg.value().decode('utf-8')))
            else:
                break  # Exit loop on error
    finally:
        consumer.close()  # Clean up: close consumer

    return feature_records  # Return the collected feature records

Take a look at the available messages

In [6]:
import pandas as pd
features_df = pd.DataFrame(fetch_all_feature_records())
features_df

,id,lag_1,lag_2,lag_6,lag_12,lag_24,rolling_mean_7,rolling_std_7,hour,day_of_week,month,temperature_forecast
0,2024-01-02 12,4659.0,4425.0,4624.0,5455.0,4412.0,4586.714286,260.285557,12.0,1.0,1.0,NaN
1,2024-01-02 12,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-9.4
2,2024-01-02 13,5124.0,4659.0,4489.0,5406.0,4490.0,4721.428571,454.427426,13.0,1.0,1.0,NaN
3,2024-01-02 13,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-10.8
4,2024-01-02 14,5567.0,5124.0,4408.0,5324.0,4613.0,4913.000000,599.588192,14.0,1.0,1.0,NaN
5,2024-01-02 14,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-12.6
6,2024-01-02 15,5830.0,5567.0,4378.0,5210.0,4724.0,5132.714286,662.215905,15.0,1.0,1.0,NaN


Get the latest complete feature record

In [7]:
latest_feature_record = features_df.groupby('id').first().dropna().sort_index(ascending=False)[:1]
latest_feature_record

,lag_1,lag_2,lag_6,lag_12,lag_24,rolling_mean_7,rolling_std_7,hour,day_of_week,month,temperature_forecast
id,,,,,,,,,,,
2024-01-02 14,5567.0,5124.0,4408.0,5324.0,4613.0,4913.0,599.588192,14.0,1.0,1.0,-12.6


Get predictions for this record

In [8]:
# Load model and run prediction
import joblib
def load_model(model_path):
  model = joblib.load(model_path)
  return model

model = load_model("models/energy_demand_model_v4.pkl")
feature_names = ['lag_1', 'lag_2', 'lag_6', 'lag_12', 'lag_24', 'rolling_mean_7', 'rolling_std_7', 'hour', 'day_of_week', 'month', 'temperature_forecast']
latest_feature_record = latest_feature_record[feature_names]
model.predict(latest_feature_record)

/usr/local/lib/python3.10/pickle.py:1718: UserWarning: [04:20:56] WARNING: /workspace/src/collective/../data/../common/error_msg.h:82: If you are loading a serialized model (like pickle in Python, RDS in R) or
configuration generated by an older version of XGBoost, please export the model by calling
`Booster.save_model` from that version first, then load it back in current version. See:

    https://xgboost.readthedocs.io/en/stable/tutorials/saving_model.html

for more details about differences between saving model and serializing.

  setstate(state)


array([5495.6904], dtype=float32)